# Zero-Trust IAM Assumed Role & S3 Data Tier Test

This notebook tests the frictionless authentication layer that auto-refreshes AWS STS assumed role credentials based on User Identity inherently tied to this Workbench. There are NO AWS keys or hardcoded user identities in this notebook.

### Step 1: Install the aws_openshift_role_assume package
Run this cell to install the library you uploaded to the workbench.

In [1]:
!pip install ./aws_openshift_role_assume --quiet

### Step 2: Dynamically Check Access Matrix
The python wrapper automatically queries the Kubernetes API to find out who spawned this notebook.
- If you logged in as **user1**: You should see `bronze` and `silver`, but NOT `gold` or `diamond`.
- If you logged in as **user2**: You should see ONLY `diamond`.

In [2]:
import os
from botocore.exceptions import ClientError
from aws_openshift_role_assume import aws
from aws_openshift_role_assume.identity import get_user

current_user = get_user()
print(f"OpenShift AI has identified you as: {current_user}")

# Initializing the auto-refreshing, user-aware boto3 client
s3 = aws.client('s3')

buckets_to_test = [
    "rh-dynamic-role-bronze-2",
    "rh-dynamic-role-silver-2",
    "rh-dynamic-role-gold-2",
    "rh-dynamic-role-diamond-2"
]

for bucket in buckets_to_test:
    print(f"\n--- Accessing {bucket} ---")
    try:
        s3.list_objects_v2(Bucket=bucket, MaxKeys=1)
        print("SUCCESS: Access Granted!")
    except ClientError as e:
        if e.response['Error']['Code'] == 'AccessDenied':
            print(f"FAILED: Access Denied (IAM Policy blocked access)")
        else:
            print(f"FAILED: {e}")


OpenShift AI has identified you as: user1

--- Accessing rh-dynamic-role-bronze-2 ---
SUCCESS: Access Granted!

--- Accessing rh-dynamic-role-silver-2 ---
SUCCESS: Access Granted!

--- Accessing rh-dynamic-role-gold-2 ---
FAILED: Access Denied (IAM Policy blocked access)

--- Accessing rh-dynamic-role-diamond-2 ---
FAILED: Access Denied (IAM Policy blocked access)
